In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os

folder_path = '/content/drive/MyDrive/t5_squad_project_16x2'
print(os.listdir(folder_path))


['train-v1.1.json', 'كود_مشروع_احدث_نسخة_32_مرة_و5مرات8_7_2025_1_مع_اختبار.ipynb']


In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import re
import random
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    TrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
import zipfile

# =============== إعدادات أساسية ===============
os.environ["WANDB_DISABLED"] = "true"  # تعطيل WandB

base_dir = "/content/drive/MyDrive/t5_squad_project_16x2"
os.makedirs(base_dir, exist_ok=True)

# تطبيع بادئة T5 (lowercase + ":" + مسافة)
_RAW_PREFIX = "question generation:"
TASK_PREFIX = _RAW_PREFIX.strip().lower()
if not TASK_PREFIX.endswith(":"):
    TASK_PREFIX += ":"
TASK_PREFIX = TASK_PREFIX + " "  # "question generation: "

# مسارات
file_path = f"{base_dir}/train-v1.1.json"
log_file_path = f"{base_dir}/training_log_16x2.txt"
out_dir = f"{base_dir}/output_results_16x2"
model_save_dir = f"{base_dir}/t5_squad_model_final_16x2"
zip_path = f"{base_dir}/t5_squad_backup_16x2.zip"

# البذور لإعادة الإنتاج
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# =============== كولباك لوج الخسارة بنفس أسلوبك ===============
class LossLoggerCallback(TrainerCallback):
    def __init__(self, log_file_path):
        self.log_file_path = log_file_path

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        if state.global_step % 1000 == 0 or state.global_step == 1:
            step = state.global_step
            msg = f"Step {step} - Train Loss: {logs.get('loss')} - Eval Loss: {logs.get('eval_loss')}"
            print(msg)
            with open(self.log_file_path, "a") as f:
                f.write(msg + "\n")

# =============== دوال تحضير البيانات ===============
def clean_text(text):
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

def remove_duplicate_questions(data):
    seen = set()
    unique_data = []
    for item in data:
        if item['cleaned_question'] not in seen:
            seen.add(item['cleaned_question'])
            unique_data.append(item)
    return unique_data

def read_and_process_squad_data(file_path, limit=None):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print("حدث خطأ:", e)
        return []

    processed_data = []
    for article in data['data']:
        for paragraph in article['paragraphs']:
            ctx = paragraph['context']
            for qa in paragraph['qas']:
                q = clean_text(qa['question'])
                answers = [clean_text(a['text']) for a in qa.get('answers', [])]
                processed_data.append({
                    'context': ctx,
                    'cleaned_question': q,
                    'cleaned_answers': answers
                })

    processed_data = remove_duplicate_questions(processed_data)
    if limit:
        processed_data = processed_data[:limit]
    return train_test_split(processed_data, test_size=0.2, random_state=42)

class SquadDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=384, target_max_len=32):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.target_max_len = target_max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        context = item['context']
        question = item['cleaned_question']

        source_text = f"{TASK_PREFIX}context: {context}"

        inputs = self.tokenizer(
            source_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        targets = self.tokenizer(
            question,
            max_length=self.target_max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'labels': targets['input_ids'].squeeze(0)
        }

# =============== تحميل النموذج والـ tokenizer ===============
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# (اختياري) إضافة البادئة كرمز خاص — يحافظ على استقرار الترميز للبادئة
added = tokenizer.add_special_tokens({"additional_special_tokens": [TASK_PREFIX.strip()]})
if added > 0:
    model.resize_token_embeddings(len(tokenizer))

# الجهاز
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# (اختياري) تقليل الذاكرة مع تدريب طويل
model.gradient_checkpointing_enable()

# =============== تحميل بيانات SQuAD وتجهيز الداتاسيت ===============
train_data, test_data = read_and_process_squad_data(file_path)
train_dataset = SquadDataset(train_data, tokenizer, max_length=384, target_max_len=32)
test_dataset  = SquadDataset(test_data,  tokenizer, max_length=384, target_max_len=32)

# =============== Data Collator للتوليد ===============
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

# =============== مقاييس التقييم (ROUGE + SacreBLEU) ===============
try:
    import evaluate, sacrebleu
    rouge_metric = evaluate.load("rouge")

    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        # في Seq2SeqTrainer، preds قد تكون tuple
        if isinstance(preds, tuple):
            preds = preds[0]

        # استبدال -100 بـ pad_token_id قبل فك الترميز
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

        pred_texts = tokenizer.batch_decode(preds, skip_special_tokens=True)
        label_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

        # تنظيف بسيط
        pred_texts = [p.strip() for p in pred_texts]
        label_texts = [l.strip() for l in label_texts]

        r = rouge_metric.compute(predictions=pred_texts, references=label_texts)
        bleu = sacrebleu.corpus_bleu(pred_texts, [label_texts]).score
        r["sacrebleu"] = bleu
        return r

except Exception as e:
    print("تحذير: evaluate/sacrebleu غير متاحين، سيتم تعطيل compute_metrics.", e)
    def compute_metrics(_):
        return {}

# =============== إعدادات التدريب ===============
# اختيار وضع الدقة المختلطة تلقائياً
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
fp16_flag = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    output_dir=out_dir,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=15,
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_dir=f"{base_dir}/logs_16x2",
    logging_strategy="steps",
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_steps=1000,
    eval_steps=1000,
    save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=fp16_flag,
    bf16=use_bf16,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    predict_with_generate=True,
    generation_max_length=32,
    generation_num_beams=4,
    report_to=[],
    seed=seed,
)

# =============== المدرِّب (Seq2SeqTrainer) ===============
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[LossLoggerCallback(log_file_path), EarlyStoppingCallback(early_stopping_patience=3)],
)

# =============== التدريب ===============
print("بدء التدريب...")
trainer.train()

# =============== الحفظ والضغط ===============
trainer.save_model(model_save_dir)
tokenizer.save_pretrained(model_save_dir)
print(f"تم حفظ النموذج في: {model_save_dir}")
print(f"سجل الخسارة محفوظ في: {log_file_path}")

with zipfile.ZipFile(zip_path, 'w') as zipf:
    for foldername, _, filenames in os.walk(model_save_dir):
        for filename in filenames:
            file_path_ = os.path.join(foldername, filename)
            arcname = os.path.relpath(file_path_, base_dir)
            zipf.write(file_path_, arcname)
    zipf.write(log_file_path, os.path.relpath(log_file_path, base_dir))
print(f"تم ضغط الملفات في: {zip_path}")


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


بدء التدريب...


Step,Training Loss,Validation Loss
1000,1.746900,0.603622
2000,1.099000,0.561083
3000,1.060900,0.539634
4000,1.035700,0.526000
5000,1.015100,0.517064
6000,1.004500,0.509849
7000,1.004700,0.504859
8000,0.987400,0.502040
9000,0.983500,0.499247
10000,0.978800,0.497379


Step 1000 - Train Loss: 1.3468999999999998 - Eval Loss: None
Step 1000 - Train Loss: None - Eval Loss: 0.6036218166351318
Step 2000 - Train Loss: 0.699 - Eval Loss: None
Step 2000 - Train Loss: None - Eval Loss: 0.5610829353332519
Step 3000 - Train Loss: 0.6608999999999999 - Eval Loss: None
Step 3000 - Train Loss: None - Eval Loss: 0.5396339654922485
Step 4000 - Train Loss: 0.6357 - Eval Loss: None
Step 4000 - Train Loss: None - Eval Loss: 0.5260003566741943
Step 5000 - Train Loss: 0.6150999999999999 - Eval Loss: None
Step 5000 - Train Loss: None - Eval Loss: 0.5170637130737304
Step 6000 - Train Loss: 0.6044999999999999 - Eval Loss: None
Step 6000 - Train Loss: None - Eval Loss: 0.5098490476608276
Step 7000 - Train Loss: 0.6046999999999999 - Eval Loss: None
Step 7000 - Train Loss: None - Eval Loss: 0.504858648777008
Step 8000 - Train Loss: 0.5874 - Eval Loss: None
Step 8000 - Train Loss: None - Eval Loss: 0.5020401239395141
Step 9000 - Train Loss: 0.5835 - Eval Loss: None
Step 9000 - T

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


تم حفظ النموذج في: /content/drive/MyDrive/t5_squad_project_16x2/t5_squad_model_final_16x2
سجل الخسارة محفوظ في: /content/drive/MyDrive/t5_squad_project_16x2/training_log_16x2.txt
تم ضغط الملفات في: /content/drive/MyDrive/t5_squad_project_16x2/t5_squad_backup_16x2.zip


#TEST

In [13]:
# -*- coding: utf-8 -*-
import os
import json
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
import re

# ================== المسارات ==================
base_dir = "."  # المجلد الحالي
model_save_dir = os.path.join(base_dir, "t5_squad_model_final_16x2")
file_path = os.path.join(base_dir, "train-v1.1.json")

# ================== تحميل النموذج والتوكنايزر ==================
tokenizer = T5Tokenizer.from_pretrained(model_save_dir)
model = T5ForConditionalGeneration.from_pretrained(model_save_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

TASK_PREFIX = "question generation: "

# ================== تنظيف النص ==================
def clean_text(text):
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

# ================== تصحيح بعض الأسماء الشائعة ==================
def fix_names(text):
    corrections = {
        "virgin marian": "Virgin Mary",
        "bronette soubirous": "Bernadette Soubirous",
        "leurdes": "Lourdes",
        "christ": "Christ",
        "basilica of the sacred heart": "Basilica of the Sacred Heart"
    }
    for wrong, right in corrections.items():
        # تصحيح بدون مراعاة حالة الأحرف
        text = re.sub(wrong, right, text, flags=re.IGNORECASE)
    return text

# ================== تجهيز بيانات الاختبار ==================
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

samples = []
for article in data['data']:
    for paragraph in article['paragraphs']:
        ctx = paragraph['context']
        samples.append(ctx)

num_contexts = 15
samples = samples[:num_contexts]

# ================== التوليد ==================
for i, context in enumerate(samples, 1):
    input_text = f"{TASK_PREFIX}context: {context}"
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=384,
        padding="max_length"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=64,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.9,
            num_return_sequences=3,
            no_repeat_ngram_size=2
        )

    gen_questions = [fix_names(tokenizer.decode(out, skip_special_tokens=True).strip()) for out in outputs]

    # اختيار أفضل سؤال (الأقرب لطول المتوسط)
    avg_len = sum(len(q.split()) for q in gen_questions) / len(gen_questions)
    best_q = min(gen_questions, key=lambda x: abs(len(x.split()) - avg_len))

    # إضافة علامة استفهام إذا لم تكن موجودة
    if not best_q.endswith('?'):
        best_q += '?'

    # ================== عرض النتائج ==================
    print(f"--- Sample {i} ---")
    print(f"Context: {context}")
    print("Generated Questions:")
    for idx, q in enumerate(gen_questions, 1):
        if not q.endswith('?'):
            q += '?'
        print(f"{idx}. {q}")
    print(f"Best Generated Question: {best_q}\n")


--- Sample 1 ---
Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.
Generated Questions:
1. what is the stone statue of morgan?
2. where did the virgin mary appear?
3. when did the virgin mary appear to saint robinette soubirous?
Best Generated Question: what is the stone statue of morgan?

--- Sample 2 ---
Context: As at most other universitie

--- Sample 8 ---
Context: The library system of the university is divided between the main library and each of the colleges and schools. The main building is the 14-story Theodore M. Hesburgh Library, completed in 1963, which is the third building to house the main collection of books. The front of the library is adorned with the Word of Life mural designed by artist Millard Sheets. This mural is popularly known as "Touchdown Jesus" because of its proximity to Notre Dame Stadium and Jesus' arms appearing to make the signal for a touchdown.
Generated Questions:
1. how many floors of the theodore m hesburgh library are there?
2. who designed the word of life mural?
3. what is the name of the mural of how much of monsoon is used for?
Best Generated Question: how many floors of the theodore m hesburgh library are there?

--- Sample 9 ---
Context: Notre Dame is known for its competitive admissions, with the incoming class enrolling in fall 2015 admitting 3,577 from a pool of 18,156 (19.7%).

--- Sample 15 ---
Context: As of 2012[update] research continued in many fields. The university president, John Jenkins, described his hope that Notre Dame would become "one of the pre–eminent research institutions in the world" in his inaugural address. The university has many multi-disciplinary institutes devoted to research in varying fields, including the Medieval Institute, the Kellogg Institute for International Studies, the Kroc Institute for International Peace studies, and the Center for Social Concerns. Recent research includes work on family conflict and child development, genome mapping, the increasing trade deficit of the United States with China, studies in fluid mechanics, computational science and engineering, and marketing trends on the Internet. As of 2013, the university is home to the Notre Dame Global Adaptation Index which ranks countries annually based on how vulnerable they are to climate change and how prepared they are to adapt.
Generated Questions:
1. what in